# Day 17：OOF 阈值稳定性、Recall/FN floor 与 Bin Projection

Day 15/16 显示，单一 validation split 上调出的参数和阈值没有稳定泛化到 official test。本轮只使用 official train 内部的 OOF / Repeated CV 预测来选择阈值，并加入 Recall/FN floor 业务约束，同时验证匿名 histogram/bin-like 前缀组是否能提供额外结构信号。

## 1. 实验边界

- 本 notebook 不使用 official test 做策略、参数或阈值选择。
- 每个 fold 内，imputer、结构特征 builder、bin projection builder 都只在 fold_train 上 fit。
- Day 17 的 OOF 最优只作为 Day 18 official test 候选，不是最终模型结论。
- Histogram/bin projection 只利用匿名字段命名结构，不解释真实物理含义。

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.models.oof_threshold_experiments import run_oof_recall_floor_bin_projection_experiments

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
with (PROJECT_ROOT / "config" / "structural_features.yaml").open("r", encoding="utf-8") as file:
    structural_config = yaml.safe_load(file)

cfg.oof_threshold["candidate_strategies"]

['baseline_median_all',
 'median_all_structural_all',
 'median_all_bin_projection',
 'median_all_structural_all_plus_bin_projection']

## 2. 读取 official train/test，但本轮只使用 train

这里读取 test 只是为了沿用统一数据入口；Day 17 不在 test 上评估，也不根据 test 结果修改任何规则。

In [2]:
train_df, _test_df = load_train_test_with_target(cfg)
train_df.shape, train_df["target"].value_counts().to_dict()

((60000, 172), {0: 59000, 1: 1000})

## 3. OOF / Repeated CV 设置

默认配置是 5 folds x 2 repeats。如果本地运行时间过长，可以在脚本层通过环境变量临时降级为 5 folds x 1 repeat，但需要在报告中说明实际运行规模。

In [3]:
cfg.oof_threshold["cv"], cfg.oof_threshold["threshold_selection_rules"]

({'n_splits': 5, 'n_repeats': 2, 'stratified': True, 'shuffle': True},
 ['cost_min',
  'recall_floor_975',
  'recall_floor_980',
  'fn_floor_20',
  'fn_floor_25'])

## 4. 运行 Day 17 OOF 实验

运行时间取决于 folds、repeats 和候选策略数量。脚本版本会保存完整 CSV 和图表；notebook 中可以先运行同一入口函数确认结果。

In [4]:
outputs = run_oof_recall_floor_bin_projection_experiments(
    train_df=train_df,
    cfg=cfg,
    structural_config=structural_config,
    oof_config=cfg.oof_threshold,
    candidate_strategies=cfg.oof_threshold["candidate_strategies"],
)

outputs.keys()

dict_keys(['oof_threshold_metrics', 'oof_best_threshold_summary', 'oof_stability_summary', 'oof_strategy_compare', 'raw_oof_predictions', 'averaged_oof_predictions', 'oof_fold_summary', 'bin_projection_metadata', 'oof_candidate_metadata', 'oof_fold_metric_summary'])

## 5. OOF threshold grid 与约束规则

`cost_min` 直接选择 OOF total_cost 最低阈值；`recall_floor_*` 和 `fn_floor_*` 是业务约束，用来降低漏报风险，而不是为了刷模型分数。

In [5]:
best_summary = outputs["oof_best_threshold_summary"]
best_summary[[
    "candidate_strategy", "threshold_selection_rule", "threshold",
    "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost",
]]

,candidate_strategy,threshold_selection_rule,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,median_all_structural_all,cost_min,0.17,0.369615,0.961,0.728030,0.873871,1639,39,35890
1,median_all_bin_projection,cost_min,0.14,0.351095,0.962,0.713650,0.872643,1778,38,36780
2,median_all_structural_all_plus_bin_projection,cost_min,0.12,0.330593,0.965,0.697355,0.873199,1954,35,37040
3,baseline_median_all,cost_min,0.12,0.324370,0.965,0.691756,0.872827,2010,35,37600
4,median_all_bin_projection,fn_floor_20,0.02,0.181264,0.981,0.521143,0.872643,4431,19,53810
5,baseline_median_all,fn_floor_20,0.02,0.172931,0.980,0.506879,0.872827,4687,20,56870
6,median_all_structural_all_plus_bin_projection,fn_floor_20,0.01,0.131386,0.985,0.428373,0.873199,6512,15,72620
7,median_all_structural_all,fn_floor_20,0.01,0.124275,0.986,0.413105,0.873871,6948,14,76480
8,median_all_structural_all_plus_bin_projection,fn_floor_25,0.05,0.249808,0.975,0.616854,0.873199,2928,25,41780
9,baseline_median_all,fn_floor_25,0.04,0.225658,0.978,0.586753,0.872827,3356,22,44560


## 6. Bin projection 信号观察

对 `ag`、`ay`、`az`、`ba`、`cn`、`cs`、`ee` 等匿名前缀组生成行级聚合特征，例如 sum、zero_rate、weighted_mean_bin、tail_ratio、peak_bin_index。它们只表示匿名分箱结构，不代表具体传感器含义。

In [6]:
outputs["bin_projection_metadata"].head(20)

,prefix,source_columns,n_bins,generated_features,keep_original_bin_columns,candidate_strategy,repeat_id,fold_id
0,ag,ag_000|ag_001|ag_002|ag_003|ag_004|ag_005|ag_0...,10,bin_ag_sum|bin_ag_mean|bin_ag_std|bin_ag_max|b...,True,median_all_bin_projection,1,1
1,ay,ay_000|ay_001|ay_002|ay_003|ay_004|ay_005|ay_0...,10,bin_ay_sum|bin_ay_mean|bin_ay_std|bin_ay_max|b...,True,median_all_bin_projection,1,1
2,az,az_000|az_001|az_002|az_003|az_004|az_005|az_0...,10,bin_az_sum|bin_az_mean|bin_az_std|bin_az_max|b...,True,median_all_bin_projection,1,1
3,ba,ba_000|ba_001|ba_002|ba_003|ba_004|ba_005|ba_0...,10,bin_ba_sum|bin_ba_mean|bin_ba_std|bin_ba_max|b...,True,median_all_bin_projection,1,1
4,cn,cn_000|cn_001|cn_002|cn_003|cn_004|cn_005|cn_0...,10,bin_cn_sum|bin_cn_mean|bin_cn_std|bin_cn_max|b...,True,median_all_bin_projection,1,1
5,cs,cs_000|cs_001|cs_002|cs_003|cs_004|cs_005|cs_0...,10,bin_cs_sum|bin_cs_mean|bin_cs_std|bin_cs_max|b...,True,median_all_bin_projection,1,1
6,ee,ee_000|ee_001|ee_002|ee_003|ee_004|ee_005|ee_0...,10,bin_ee_sum|bin_ee_mean|bin_ee_std|bin_ee_max|b...,True,median_all_bin_projection,1,1
7,ag,ag_000|ag_001|ag_002|ag_003|ag_004|ag_005|ag_0...,10,bin_ag_sum|bin_ag_mean|bin_ag_std|bin_ag_max|b...,True,median_all_bin_projection,1,2
8,ay,ay_000|ay_001|ay_002|ay_003|ay_004|ay_005|ay_0...,10,bin_ay_sum|bin_ay_mean|bin_ay_std|bin_ay_max|b...,True,median_all_bin_projection,1,2
9,az,az_000|az_001|az_002|az_003|az_004|az_005|az_0...,10,bin_az_sum|bin_az_mean|bin_az_std|bin_az_max|b...,True,median_all_bin_projection,1,2


## 7. OOF 稳定性分析

这里观察每个候选策略在不同 fold/repeat 下的 recall、FN 和 total_cost 波动。若某个策略 OOF 均值好但波动大，Day 18 进入 official test 时要谨慎。

In [7]:
outputs["oof_stability_summary"].sort_values(["threshold_selection_rule", "mean_cost"]).head(20)

,candidate_strategy,threshold_selection_rule,selected_threshold,mean_cost,std_cost,mean_fn,std_fn,mean_recall,std_recall,mean_precision,std_precision,mean_f2,std_f2
0,baseline_median_all,cost_min,0.12,7990.0,1018.168290,8.1,2.424413,0.9595,0.012122,0.327903,0.010130,0.692394,0.005287
5,median_all_bin_projection,cost_min,0.14,8017.0,1317.700438,9.0,2.867442,0.9550,0.014337,0.352182,0.008296,0.711267,0.006335
15,median_all_structural_all_plus_bin_projection,cost_min,0.12,8079.0,1228.254860,8.5,2.677063,0.9575,0.013385,0.333876,0.012637,0.696785,0.010478
10,median_all_structural_all,cost_min,0.17,8255.0,1099.679246,10.0,2.449490,0.9500,0.012247,0.368780,0.007506,0.722161,0.003296
6,median_all_bin_projection,fn_floor_20,0.02,11043.0,826.680779,4.6,1.429841,0.9770,0.007149,0.182928,0.007398,0.522785,0.012417
1,baseline_median_all,fn_floor_20,0.02,11734.0,411.668691,4.9,1.197219,0.9755,0.005986,0.173959,0.007379,0.507427,0.011657
16,median_all_structural_all_plus_bin_projection,fn_floor_20,0.01,14806.0,705.032702,3.7,1.337494,0.9815,0.006687,0.131767,0.005100,0.428501,0.010667
11,median_all_structural_all,fn_floor_20,0.01,15681.0,814.772634,3.7,0.823273,0.9815,0.004116,0.124743,0.007709,0.413154,0.016688
17,median_all_structural_all_plus_bin_projection,fn_floor_25,0.05,8760.0,749.325623,6.0,1.825742,0.9700,0.009129,0.252205,0.007808,0.617950,0.008053
12,median_all_structural_all,fn_floor_25,0.04,9362.0,840.235417,5.6,1.837873,0.9720,0.009189,0.228744,0.006882,0.588993,0.008946


## 8. Day 18 候选建议

Day 18 只能固定 Day 17 在 OOF 上选出的 1-2 个候选策略和阈值，在 official test 上做一次最终观察。不能在 test 上重新选阈值，也不能根据 test 结果回头修改 OOF 规则。